# Program 19
## Next Word Prediction using LSTM and GRU, Compare the performance of both
Uses a small NLTK corpus, creates sequences, and trains both an LSTM and a GRU model to predict the next word.

In [ ]:
import nltk
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Embedding
import time

# 1. Load and clean data from NLTK
nltk.download('gutenberg')
from nltk.corpus import gutenberg

print("Loading dataset...")
# Taking a small subset of words to train in reasonable time
words = gutenberg.words('blake-poems.txt')[:2000]
text = " ".join([w.lower() for w in words if w.isalpha()])

# 2. Tokenize the data
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# 3. Create increasing sequences
input_sequences = []
token_list = tokenizer.texts_to_sequences([text])[0]
for i in range(1, len(token_list)):
    n_gram_sequence = token_list[max(0, i-5):i+1] # contextual window up to 5 words
    input_sequences.append(n_gram_sequence)

print("Sample sequences:", input_sequences[:5])

# 4 & 5. Apply padding to make sequences same length, Create X and y
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

X, labels = input_sequences[:,:-1], input_sequences[:,-1]
y = to_categorical(labels, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

# 6. Apply embedding and the LSTM and GRU layers
def build_model(rnn_type):
    model = Sequential()
    model.add(Embedding(total_words, 20, input_length=max_sequence_len-1))
    if rnn_type == 'LSTM':
        model.add(LSTM(50))
    else:
        model.add(GRU(50))
    model.add(Dense(total_words, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

lstm_model = build_model('LSTM')
gru_model = build_model('GRU')

# Train Models
print("\nTraining LSTM Model...")
lstm_start = time.time()
history_lstm = lstm_model.fit(X, y, epochs=50, verbose=0)
lstm_time = time.time() - lstm_start

print("Training GRU Model...")
gru_start = time.time()
history_gru = gru_model.fit(X, y, epochs=50, verbose=0)
gru_time = time.time() - gru_start

# 7. Evaluate Models
lstm_acc = history_lstm.history['accuracy'][-1]
gru_acc = history_gru.history['accuracy'][-1]

print(f"\nLSTM Final Training Accuracy: {lstm_acc*100:.2f}% (Time: {lstm_time:.2f}s)")
print(f"GRU Final Training Accuracy: {gru_acc*100:.2f}% (Time: {gru_time:.2f}s)")

# 8. Predict next word
def predict_next_word(model, text_input):
    sequence = tokenizer.texts_to_sequences([text_input])[0]
    sequence = pad_sequences([sequence], maxlen=max_sequence_len-1, padding='pre')
    predicted_prob = model.predict(sequence, verbose=0)
    predicted_class = np.argmax(predicted_prob, axis=-1)[0]
    for word, index in tokenizer.word_index.items():
        if index == predicted_class:
            return word
    return ""

seed_text = "the young child"
print(f"\nSeed Text: '{seed_text}'")
print(f"LSTM Next Word: '{predict_next_word(lstm_model, seed_text)}'")
print(f"GRU Next Word: '{predict_next_word(gru_model, seed_text)}'")

# 9. Inference
print("\nInference:")
print("GRU usually trains faster than LSTM due to fewer gates.")
print("Both models attempt to learn the local contextual sequence. Their accuracy varies based on convergence.")